# Exercise 3: Hyperparameter Optimization

In this exercise we tune the workflow for better quality and efficiency.

## What We're Optimizing

| Parameter | Range | Impact |
|-----------|-------|--------|
| `temperature` | 0.0 - 1.0 | Creativity vs. determinism |
| `max_tokens` | 256 - 4096 | Output length vs. cost |

**Goal:** Find the sweet spot between answer quality, speed, and token cost.

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../../.env")

client = OpenAI(
    base_url=os.getenv("VLLM_BASE_URL"),
    api_key=os.getenv("VLLM_API_KEY"),
)
MODEL = os.getenv("VLLM_MODEL_NAME")

## Step 1: Review the Evaluation Dataset

Optimization needs a benchmark - questions with known answers.

In [ ]:
with open("eval_dataset.json") as f:
    dataset = json.load(f)

print(f"Evaluation dataset: {len(dataset)} questions\n")
for item in dataset[:3]:
    print(f"  Q: {item['question']}")
    print(f"  A: {item['answer']}")
    print()

## Step 2: Temperature Sweep

How does temperature affect correctness, token usage, and latency?

In [ ]:
test_question = "W którym roku Polska przystąpiła do Unii Europejskiej?"
expected = "2004"

print(f"Question: {test_question}")
print(f"Expected keyword: {expected}")
print("=" * 60)

for temp in [0.0, 0.3, 0.5, 0.7, 1.0]:
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": test_question}],
        temperature=temp,
        max_tokens=128,
    )
    latency = (time.time() - start) * 1000
    answer = response.choices[0].message.content.strip()
    tokens = response.usage.total_tokens
    correct = expected.lower() in answer.lower()
    
    print(f"\ntemp={temp:.1f} | {latency:.0f}ms | {tokens} tokens | {'CORRECT' if correct else 'WRONG'}")
    print(f"  -> {answer[:120]}")

## Step 3: Compare Configurations Across Multiple Questions

In [ ]:
configs = {
    "conservative": {"temperature": 0.1, "max_tokens": 256},
    "balanced":     {"temperature": 0.5, "max_tokens": 512},
    "default":      {"temperature": 0.7, "max_tokens": 1024},
    "creative":     {"temperature": 1.0, "max_tokens": 1024},
}

eval_questions = [
    ("Jaka jest stolica Polski?", "warszaw"),
    ("Kiedy Polska przystąpiła do UE?", "2004"),
    ("Kto napisał Pana Tadeusza?", "mickiewicz"),
]

print(f"{'Config':<15} {'Correct':>8} {'Avg Tokens':>11} {'Avg Latency':>12} {'Est. Cost':>10}")
print("-" * 60)

# Beyond pricing
INPUT_RATE = 0.10 / 1_000_000
OUTPUT_RATE = 0.20 / 1_000_000

for name, params in configs.items():
    correct = 0
    total_in = 0
    total_out = 0
    total_latency = 0
    
    for question, expected_keyword in eval_questions:
        start = time.time()
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": question}],
            **params,
        )
        latency = (time.time() - start) * 1000
        answer = response.choices[0].message.content
        
        if expected_keyword.lower() in answer.lower():
            correct += 1
        total_in += response.usage.prompt_tokens
        total_out += response.usage.completion_tokens
        total_latency += latency
    
    n = len(eval_questions)
    cost = total_in * INPUT_RATE + total_out * OUTPUT_RATE
    print(f"{name:<15} {correct}/{n:>6} {(total_in+total_out)/n:>10.0f} {total_latency/n:>10.0f}ms ${cost:>.6f}")

## Step 4: NAT Optimizer (Demo)

NAT includes an Optuna-based optimizer that systematically searches parameter space.

```bash
nat optimize --config_file config.yaml --dataset eval_dataset.json
```

This takes several minutes to run. It will try different combinations of
temperature and other optimizable parameters, evaluate against the dataset,
and report the best configuration found.

In [ ]:
# Uncomment to run (takes several minutes):
# !nat optimize --config_file config.yaml --dataset eval_dataset.json

## Step 5: Quantify Optimization Impact on Cost

Lower temperature -> shorter, more focused answers -> fewer tokens -> lower cost.

In [ ]:
# Run the same question with different max_tokens to see cost impact
question = "Czym jest Kopalnia Soli w Wieliczce?"

print(f"Question: {question}\n")
print(f"{'max_tokens':<12} {'Output':>8} {'Cost':>10} {'Answer preview'}")
print("-" * 70)

for max_tok in [64, 128, 256, 512, 1024]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0.5,
        max_tokens=max_tok,
    )
    out_tokens = response.usage.completion_tokens
    cost = response.usage.prompt_tokens * INPUT_RATE + out_tokens * OUTPUT_RATE
    answer = response.choices[0].message.content[:60].replace('\n', ' ')
    print(f"{max_tok:<12} {out_tokens:>8} ${cost:>8.6f}   {answer}...")

print(f"\nKey insight: max_tokens controls cost ceiling per call.")
print(f"A workflow with 7 LLM calls at max_tokens=1024 vs 256: ~4x cost difference.")

## Part 3 Summary

### What We Learned

1. **Observability** (Exercise 1)
   - Phoenix traces every LLM call automatically
   - Traces reveal bottlenecks and failure patterns

2. **Cost Tracking** (Exercise 2)
   - Token Factory pricing: `(input x rate) + (output x rate)`
   - Per-agent tracking shows where the money goes
   - Output tokens are the main cost driver

3. **Optimization** (Exercise 3)
   - Temperature affects correctness and verbosity
   - `max_tokens` directly controls cost ceiling
   - `nat optimize` automates parameter search

### Production Checklist

- [ ] Tracing enabled for all workflows
- [ ] Per-agent cost tracking dashboards
- [ ] Evaluation dataset maintained and expanded
- [ ] Parameters optimized per use case
- [ ] Cost alerts and per-user budgets